> **Ekip paylaşım kopyası:** Kayıtlı çıktılar ve widget durumu dosya boyutunu azaltmak için temizlenmiştir; analiz hücreleri korunur. Veri dosyası Git deposuna dahil değildir. Kurulum, veri yolu ve mevcut veri dosyasını kopyalamadan kullanma seçenekleri için `../README.md` dosyasına bakın. `.venv` kernel'ini seçip Run All çalıştırın. Kaynak projedeki notebook ve çıktılar değiştirilmemiştir.

# DS02 — altı dev motorunun yaşam döngüsü karşılaştırması
**Amaç:** Motor 2, 5, 10, 16, 18 ve 20'nin uçuş sürelerini, sensör davranışlarını, simülasyon iç sağlık parametrelerini ve çalışma koşullarını bütün ilgili ölçümler üzerinden incelemek.

**Ana EDA kapsamı:** hiçbir uçuş veya çalışma rejimi koşul filtresiyle dışlanmaz. Önceki notebook'taki benzer koşul alt grubu burada uygulanmaz. Özetler bütün uçuş ölçümlerinden hesaplanır; tüm DS02 tek DataFrame'e alınmaz. HDF5 yalnızca `r` modunda açılır. Test verileri okunmaz; yalnızca dev dizileri ve değişken isimlerini taşıyan metadata kullanılır.

**Sınırlar:** T_dev sensör değil simülasyonun iç sağlık durumudur. İşaret ve ölçek korunur; parametreler doğrudan verim kaybı yüzdesi değildir. Sıfırdan farklı ilk değer arıza başlangıcı değildir. RUL'nin tanım gereği düşmesi bir model başarısı değildir. Bu notebook modelleme, özellik seçimi veya preprocessing yapmaz.

**Kullanım:** VS Code'da proje `.venv` Jupyter kernel'ini seçip Run All çalıştırın. Son bölümde sensör seçimi grafiklerini yeniden HDF5 okumadan değiştirir. Yeni CSV klasörü her çalıştırmada ayrı oluşturulur; eski sonuçların üzerine yazılmaz.

In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import ipywidgets as widgets
import os
import sys
# Matplotlib ?nbelle?i proje sanal ortam?nda tutulur.
os.environ.setdefault("MPLCONFIGDIR", str(Path(__import__("tempfile").gettempdir()) / "n_cmapss_matplotlib_cache"))
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Paylaşım kopyası: veri yolu bilgisayardan bağımsız çözülür.
import os

def resolve_ncmapss_paths(start=None):
    working_directory = Path.cwd().resolve() if start is None else Path(start).resolve()
    project_root = None
    for parent in [working_directory, *working_directory.parents]:
        if parent.name == "n_cmapss" and (parent / "notebooks").is_dir():
            project_root = parent
            break
        candidate = parent / "n_cmapss"
        if (candidate / "notebooks").is_dir():
            project_root = candidate
            break
    if project_root is None:
        raise FileNotFoundError("Notebook'u ortak repo, n_cmapss veya n_cmapss/notebooks klasöründen çalıştırın.")
    external_path = os.environ.get("N_CMAPSS_H5")
    h5_path = Path(external_path).expanduser().resolve() if external_path else project_root / "data" / "N-CMAPSS_DS02-006.h5"
    return project_root, h5_path

project_root, h5_path = resolve_ncmapss_paths()
if not h5_path.is_file():
    raise FileNotFoundError(
        f"Veri dosyası bulunamadı: {h5_path}. HDF5 repoya dahil değildir. "
        "Dosyayı yerel n_cmapss/data/ altında sağlayın veya kernel ortamında "
        "N_CMAPSS_H5 değişkenini mevcut dosyanın tam yoluna ayarlayın. README'ye bakın."
    )

DEV_KEYS = ["A_dev", "W_dev", "X_s_dev", "X_v_dev", "T_dev", "Y_dev"]
VAR_KEYS = ["A_var", "W_var", "X_s_var", "X_v_var", "T_var"]
CHUNK_ROWS = 100_000

def decode_names(dataset):
    return [v.decode("utf-8").rstrip("\x00") if isinstance(v, (bytes, np.bytes_)) else str(v)
            for v in np.asarray(dataset[()]).reshape(-1)]

with h5py.File(h5_path, "r") as h5:
    names = {key: decode_names(h5[key]) for key in VAR_KEYS}
    column_names = {key: names[var] for key, var in zip(DEV_KEYS[:-1], VAR_KEYS)}
    column_names["Y_dev"] = ["RUL"]
    row_counts = {key: h5[key].shape[0] for key in DEV_KEYS}
    if len(set(row_counts.values())) != 1:
        raise ValueError(f"Satır sayıları eşit değil: {row_counts}")
    for key in DEV_KEYS:
        if h5[key].ndim != 2 or h5[key].shape[1] != len(column_names[key]):
            raise ValueError(f"Sütun metadata uyumsuzluğu: {key}")
all_columns = [name for key in DEV_KEYS for name in column_names[key]]
assert len(all_columns) == len(set(all_columns)) == 47
unit_col = names["A_var"].index("unit")
cycle_col = names["A_var"].index("cycle")
print("dev satır sayısı:", row_counts["A_dev"])
from IPython.display import HTML, Markdown
from collections import Counter
import json

%matplotlib inline
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.grid": True, "grid.alpha": 0.25})


## 1. Bellek verimli okuma ve satır hizalaması
A_dev içindeki unit/cycle değerleri 100.000 satırlık parçalarla taranır; yalnızca uçuş satır aralıkları saklanır. Altı dev dizisinin satır sayıları ve metadata genişlikleri önceden doğrulanır. Her uçuş aynı artan HDF5 aralıklarıyla okunur; kimlik ve ölçüm sayısı kontrolleri yapılır. Bellekte bir uçuşun 47 değişkeni ve 446 uçuşun küçük özetleri tutulur; motorların bütün ölçümleri biriktirilmez.

NaN/sonsuz değer varsa sayılır ve istatistik hesabı için geçici olarak NaN kabul edilir; orijinal veri değiştirilmez. Bu durum ayrı raporlanır. Sabitlik, uçuş medyanlarına değil bütün ilgili ölçümlerin min–maksimumuna göre kontrol edilir.

In [ ]:
flight_ranges = {}
with h5py.File(h5_path, "r") as h5:
    a = h5["A_dev"]
    for start in range(0, a.shape[0], CHUNK_ROWS):
        stop = min(start + CHUNK_ROWS, a.shape[0])
        units = a[start:stop, unit_col]
        cycles = a[start:stop, cycle_col]
        if not (np.isfinite(units).all() and np.isfinite(cycles).all()
                and np.equal(units, np.floor(units)).all()
                and np.equal(cycles, np.floor(cycles)).all()):
            raise ValueError("Motor/uçuş kimlikleri sonlu tam sayı olmalıdır")
        changes = np.flatnonzero((units[1:] != units[:-1]) | (cycles[1:] != cycles[:-1])) + 1
        boundaries = np.r_[0, changes, len(units)]
        for left, right in zip(boundaries[:-1], boundaries[1:]):
            key = (int(units[left]), int(cycles[left]))
            begin, end = start + int(left), start + int(right)
            ranges = flight_ranges.setdefault(key, [])
            if ranges and ranges[-1][1] == begin:
                ranges[-1] = (ranges[-1][0], end)
            else:
                ranges.append((begin, end))
engine_flights = {}
for unit, cycle in sorted(flight_ranges):
    engine_flights.setdefault(unit, []).append(cycle)
assert sum(end - begin for ranges in flight_ranges.values() for begin, end in ranges) == row_counts["A_dev"]
catalog = pd.DataFrame([
    {"unit": unit, "uçuş_sayısı": len(cycles), "ilk_cycle": min(cycles), "son_cycle": max(cycles)}
    for unit, cycles in engine_flights.items()
])
display(catalog)
print("Toplam motor/uçuş çifti:", len(flight_ranges))

assert list(engine_flights) == [2, 5, 10, 16, 18, 20]


## 2. Bütün ölçümlerden uçuş ve motor özetleri
Her sensör ve koşul için medyan/Q1/Q3; sağlık parametreleri için min/medyan/maksimum hesaplanır. İlk/son değer ifadesi ilk/son **uçuş medyanını** belirtir; tek bir ham satır değildir. RUL'nin uçuş içi sabitliği ve min–maksimumu kaydedilir. Fc/hs dağılımları ölçüm sayısıyla ağırlıklı ve uçuş kapsamıyla ayrı raporlanır; farklı Fc sınıflarında uçuş uzunlukları farklı olabileceğinden bu iki sayı aynı kavram değildir.

In [ ]:
ENGINES = [2, 5, 10, 16, 18, 20]
sensors = names["X_s_var"] + names["X_v_var"]
physical_sensors, virtual_sensors = names["X_s_var"], names["X_v_var"]
health_parameters = names["T_var"]
conditions = ["alt", "Mach", "TRA", "T2"]
assert len(sensors) == 28 and len(health_parameters) == 10
assert list(engine_flights) == ENGINES

# İstatistikler uçuş bazında; referans kutusu veya ölçüm filtresi yok.
flight_rows, health_rows, distribution_rows, quality_rows, issue_rows = [], [], [], [], []
with h5py.File(h5_path, "r") as h5:
    for unit in ENGINES:
        limits = {name: [np.inf, -np.inf, 0, 0] for name in all_columns}
        first_nonzero = {name: None for name in health_parameters}
        nonzero_counts = Counter()
        class_counts = {name: Counter() for name in ["Fc", "hs"]}
        class_flights = {name: Counter() for name in ["Fc", "hs"]}
        for cycle in sorted(engine_flights[unit]):
            ranges = flight_ranges[(unit, cycle)]
            frames = []
            for key in DEV_KEYS:
                blocks = [h5[key][begin:end, :] for begin, end in ranges]
                array = blocks[0] if len(blocks)==1 else np.concatenate(blocks, axis=0)
                frames.append(pd.DataFrame(array, columns=column_names[key]))
            data = pd.concat(frames, axis=1, verify_integrity=True)
            expected_n = sum(end-begin for begin,end in ranges)
            assert len(data) == expected_n
            assert (data.unit == unit).all() and (data.cycle == cycle).all()
            row = {"unit":unit, "cycle":cycle, "n_samples":len(data)}
            mask = np.isfinite(data.to_numpy())
            row["nonfinite_count"] = int((~mask).sum())
            if row["nonfinite_count"]:
                issue_rows.append({"unit":unit,"cycle":cycle,"issue":"Sonlu olmayan değer", "count":row["nonfinite_count"]})
            clean = data.where(mask, np.nan)
            minimums, maximums = clean.min(), clean.max()
            for name in all_columns:
                state = limits[name]
                lo,hi = minimums[name],maximums[name]
                if pd.notna(lo): state[0] = min(state[0],lo)
                if pd.notna(hi): state[1] = max(state[1],hi)
                state[2] += int(clean[name].isna().sum())
                state[3] += int(clean[name].notna().sum())
            rul=clean.RUL
            row.update(RUL_first=rul.iloc[0],RUL_median=rul.median(),RUL_min=rul.min(),RUL_max=rul.max(),
                       RUL_constant=bool(rul.notna().all() and rul.nunique()==1))
            if not row["RUL_constant"]:
                issue_rows.append({"unit":unit,"cycle":cycle,"issue":"RUL uçuş içinde sabit/eksiksiz değil","count":1})
            for name in ["Fc","hs"]:
                counts=clean[name].value_counts().sort_index()
                row[name+"_counts"] = json.dumps({str(k):int(v) for k,v in counts.items()},ensure_ascii=False)
                row[name+"_nunique"] = len(counts)
                for value,count in counts.items():
                    class_counts[name][float(value)] += int(count)
                    class_flights[name][float(value)] += 1
                if len(counts)>1:
                    issue_rows.append({"unit":unit,"cycle":cycle,"issue":name+" uçuş içinde değişiyor (otomatik hata değil)","count":len(counts)})
            quantiles=clean[sensors+conditions].quantile([0.25,0.5,0.75])
            for name in sensors+conditions:
                for q,label in [(0.25,"q1"),(0.5,"median"),(0.75,"q3")]:
                    row[name+"_"+label]=quantiles.loc[q,name]
            health_medians=clean[health_parameters].median()
            for name in health_parameters:
                row[name+"_min"],row[name+"_median"],row[name+"_max"] = minimums[name],health_medians[name],maximums[name]
                count=int((clean[name].notna() & clean[name].ne(0)).sum())
                row[name+"_nonzero_n"]=count
                nonzero_counts[name]+=count
                if count and first_nonzero[name] is None: first_nonzero[name]=cycle
            flight_rows.append(row)
            del data,clean,frames,blocks,array
        for name in ["Fc","hs"]:
            for value,count in sorted(class_counts[name].items()):
                distribution_rows.append({"unit":unit,"variable":name,"value":value,"n_measurements":count,
                                          "n_flights_containing_value":class_flights[name][value]})
        for name,(lo,hi,missing,valid) in limits.items():
            status = "Tamamı eksik" if valid==0 else ("Sabit sıfır" if lo==hi==0 else ("Sabit sıfır dışı" if lo==hi else "Değişken"))
            quality_rows.append({"unit":unit,"variable":name,"min":lo if valid else np.nan,"max":hi if valid else np.nan,
                                 "nonfinite_count":missing,"n_valid":valid,"status":status})
        for name in health_parameters:
            lo,hi,missing,valid=limits[name]
            status = "Tamamı eksik" if valid==0 else ("Sabit sıfır" if lo==hi==0 else ("Sabit sıfır dışı" if lo==hi else "Değişken"))
            health_rows.append({"unit":unit,"parameter":name,"status":status,"nonzero_samples":nonzero_counts[name],
                               "first_nonzero_cycle":first_nonzero[name],"global_min":lo if valid else np.nan,
                               "global_max":hi if valid else np.nan,"nonfinite_count":missing})
        print(f"Motor {unit}: {len(engine_flights[unit])} uçuş hesaplandı.")

flight_summary=pd.DataFrame(flight_rows).sort_values(["unit","cycle"]).reset_index(drop=True)
health_summary=pd.DataFrame(health_rows)
health_summary["first_nonzero_cycle"]=health_summary.first_nonzero_cycle.astype("Int64")
class_distribution=pd.DataFrame(distribution_rows)
quality_summary=pd.DataFrame(quality_rows)
issues=pd.DataFrame(issue_rows,columns=["unit","cycle","issue","count"])
engine_rows=[]
for unit in ENGINES:
    s=flight_summary.loc[flight_summary.unit==unit]
    last_cycle,first_cycle=int(s.cycle.max()),int(s.cycle.min())
    flight_summary.loc[s.index,"life_progress"]=(s.cycle-first_cycle)/(last_cycle-first_cycle) if last_cycle>first_cycle else 0.0
    active=health_summary.loc[(health_summary.unit==unit)&(health_summary.nonzero_samples>0),"parameter"].tolist()
    endpoints={}
    for name in health_parameters:
        health_summary.loc[(health_summary.unit==unit)&(health_summary.parameter==name),"first_flight_median"]=s[name+"_median"].iloc[0]
        health_summary.loc[(health_summary.unit==unit)&(health_summary.parameter==name),"last_flight_median"]=s[name+"_median"].iloc[-1]
        if name in active: endpoints[name]={"first_median":s[name+"_median"].iloc[0],"last_median":s[name+"_median"].iloc[-1]}
    fc=class_distribution.loc[(class_distribution.unit==unit)&(class_distribution.variable=="Fc")]
    hs=class_distribution.loc[(class_distribution.unit==unit)&(class_distribution.variable=="hs")]
    engine_rows.append({"unit":unit,"first_cycle":first_cycle,"last_cycle":last_cycle,"n_flights":len(s),
        "n_measurements":int(s.n_samples.sum()),"mean_measurements_per_flight":s.n_samples.mean(),
        "first_RUL":s.RUL_median.iloc[0],"last_RUL":s.RUL_median.iloc[-1],
        "Fc_classes":", ".join(str(v) for v in fc.value),
        "hs_measurement_counts":json.dumps({str(r.value):int(r.n_measurements) for r in hs.itertuples()}),
        "nonzero_health_parameters":", ".join(active),"health_endpoint_medians":json.dumps(endpoints),
        "RUL_constant_flights":int(s.RUL_constant.sum())})
    if (s.RUL_median.diff().dropna()>0).any():
        issues.loc[len(issues)]=[unit,np.nan,"RUL artışı",1]
    if len(s)>1 and np.any(np.diff(s.cycle)!=1):
        issues.loc[len(issues)]=[unit,np.nan,"Cycle boşluğu",1]
engine_summary=pd.DataFrame(engine_rows)
# Bütün dev uçuşları, bütün ölçümler ve dağılımlar kapsanmalıdır.
assert engine_summary.n_measurements.sum()==row_counts["A_dev"]==5_263_447
assert len(flight_summary)==len(flight_ranges)==446
assert not flight_summary.duplicated(["unit","cycle"]).any()
for unit in ENGINES:
    expected=sum(end-begin for (u,c),ranges in flight_ranges.items() if u==unit for begin,end in ranges)
    actual=int(engine_summary.loc[engine_summary.unit==unit,"n_measurements"].iloc[0])
    assert actual==expected
    for name in ["Fc","hs"]:
        assert class_distribution.loc[(class_distribution.unit==unit)&(class_distribution.variable==name),"n_measurements"].sum()+quality_summary.loc[(quality_summary.unit==unit)&(quality_summary.variable==name),"nonfinite_count"].iloc[0]==actual
print("Kapsam doğrulandı: 6 motor, 446 uçuş, 5.263.447 dev ölçümü.")


## 3. Genel karşılaştırma, dağılımlar ve veri kalitesi
Motor ömrü uçuş sayısıyla, veri hacmi ölçüm sayısıyla ifade edilir. Bir uçuş birçok ölçüm içerir ve uçuşların ölçüm sayıları farklı olabilir; daha fazla satır tek başına daha uzun motor ömrü anlamına gelmez.

Aşağıdaki kalite tablosu bütün değişkenler üzerinde sonlu olmayan değer ve tam sabitlik kontrolünü özetler. Unit gibi kimlik alanlarının sabit olması beklenir; sabit sağlık parametreleri otomatik elenmez. Sensörlere önem sırası verilmez. Fc/hs sayılarında ölçüm ağırlığı ile uçuş kapsamı ayrı sütunlardır; uçuş içinde birden çok değer varsa kapsam toplamları uçuş sayısını aşabilir.

In [ ]:
def scroll_table(table,height=420):
    display(HTML(f"<div style='width:100%;height:{height}px;overflow:scroll'>"+table.to_html(index=False,max_rows=None,max_cols=None,escape=True)+"</div>"))
scroll_table(engine_summary)
scroll_table(health_summary,500)
scroll_table(class_distribution)
quality_findings=quality_summary.loc[(quality_summary.nonfinite_count>0)|(quality_summary.status!="Değişken")]
scroll_table(quality_findings)
if issues.empty:
    print("RUL/Fc/hs/cycle kontrollerinde beklenmeyen durum bulunmadı.")
else:
    scroll_table(issues)
print("Sonlu olmayan değer sayısı:",int(quality_summary.nonfinite_count.sum()))


## 4. Uçuş ömürleri, ölçüm hacmi ve RUL
Gerçek cycle ve görsel karşılaştırma amaçlı yaşam ilerlemesi ayrı gösterilir. İlerleme `(cycle − ilk cycle) / (son cycle − ilk cycle)` olarak hesaplanır; bütün motorlar 0–1 aralığına yerleşir. **Bu oran fiziksel bozulma ölçüsü veya model girdisi değildir.** Ömrün son cycle'ını kullandığından bu gösterim ileride tahmin modeline doğrudan taşınamaz; burada yalnızca tamamlanmış yaşamların geriye dönük görsel hizalamasıdır.

In [ ]:
engine_colors={unit:plt.get_cmap("tab10")(i) for i,unit in enumerate(ENGINES)}
engine_styles={unit:style for unit,style in zip(ENGINES,["-","--","-.",":","-","--"])}
def finish(fig):
    try: display(fig)
    finally: plt.close(fig)

def engine_series(unit):
    return flight_summary.loc[flight_summary.unit==unit].sort_values("cycle")

def count_figure():
    fig,axes=plt.subplots(1,2,figsize=(14,5),layout="constrained")
    for ax,column,title,ylabel in [(axes[0],"n_flights","Motor ömürleri","Toplam uçuş sayısı"),(axes[1],"n_measurements","Dev ölçümlerinin dağılımı","Toplam ölçüm sayısı")]:
        ax.bar(engine_summary.unit.astype(str),engine_summary[column],color=[engine_colors[u] for u in ENGINES])
        ax.set(title=title,xlabel="Motor ID",ylabel=ylabel)
        for i,value in enumerate(engine_summary[column]): ax.text(i,value,f"{int(value):,}",ha="center",va="bottom",fontsize=9)
        ax.margins(y=.15)
    return fig

def rul_figure():
    fig,axes=plt.subplots(2,1,figsize=(13,9),layout="constrained")
    for ax,x,label in [(axes[0],"cycle","cycle — uçuş döngüsü"),(axes[1],"life_progress","Normalize yaşam ilerlemesi (yalnızca görsel karşılaştırma)")]:
        for unit in ENGINES:
            s=engine_series(unit)
            ax.plot(s[x],s.RUL_median,label=f"Motor {unit}",color=engine_colors[unit],linestyle=engine_styles[unit])
            if not s.RUL_constant.all(): ax.fill_between(s[x],s.RUL_min,s.RUL_max,color=engine_colors[unit],alpha=.1)
        ax.set(xlabel=label,ylabel="RUL — kalan uçuş döngüsü")
        ax.legend(ncol=3)
    fig.suptitle("DS02 dev — RUL'nin tanım gereği azalışı (model başarısı değil)")
    return fig
finish(count_figure())
finish(rul_figure())


## 5. İç sağlık parametreleri: durum matrisi ve aktif eğriler
Durum, **tüm ham uçuş ölçümlerinin min–maksimumuna** göre belirlenir: sabit sıfır, değişken, sabit sıfır dışı veya tamamı eksik. Sıfırdan farklı olmak ile değişken olmak ayrı kavramlardır. Isı haritası şiddet/önem sıralaması değil kategorik durum matrisidir; eksik değer sayıları sağlık tablosundadır. Yalnızca en az bir motorda sıfırdan farklı olan parametrelere eğri paneli ayrılır. İşaret ve özgün ölçek korunur.

In [ ]:
def health_matrix_figure():
    from matplotlib.colors import ListedColormap, BoundaryNorm
    codes={"Sabit sıfır":0,"Değişken":1,"Sabit sıfır dışı":2,"Tamamı eksik":3}
    short={0:"Sıfır",1:"Değişken",2:"Sabit ≠0",3:"Eksik"}
    matrix=health_summary.pivot(index="unit",columns="parameter",values="status").reindex(index=ENGINES,columns=health_parameters)
    values=matrix.apply(lambda col:col.map(codes)).to_numpy()
    fig,ax=plt.subplots(figsize=(17,6),layout="constrained")
    cmap=ListedColormap(["#e8e8e8","#8fb9e3","#edc677","#d68181"])
    ax.imshow(values,cmap=cmap,norm=BoundaryNorm(np.arange(-.5,4.5),cmap.N),aspect="auto")
    ax.set_xticks(range(10),health_parameters,rotation=35,ha="right")
    ax.set_yticks(range(6),[f"Motor {u}" for u in ENGINES])
    for i in range(6):
        for j in range(10): ax.text(j,i,short[values[i,j]],ha="center",va="center",fontsize=9)
    ax.set_title("DS02 dev — 10 iç sağlık parametresinin kategorik durumu (şiddet haritası değil)")
    return fig

active_parameters=[p for p in health_parameters if health_summary.loc[health_summary.parameter==p,"nonzero_samples"].sum()>0]
def active_health_figure():
    count=max(1,len(active_parameters))
    fig,axes=plt.subplots(count,1,figsize=(14,4*count),squeeze=False,layout="constrained")
    if not active_parameters: axes[0,0].text(.5,.5,"Sıfırdan farklı sağlık parametresi bulunmadı",ha="center")
    for ax,name in zip(axes[:,0],active_parameters):
        for unit in ENGINES:
            h=health_summary.loc[(health_summary.unit==unit)&(health_summary.parameter==name)].iloc[0]
            if not h.nonzero_samples: continue
            s=engine_series(unit)
            ax.plot(s.cycle,s[name+"_median"],color=engine_colors[unit],linestyle=engine_styles[unit],label=f"Motor {unit}")
            ax.fill_between(s.cycle,s[name+"_min"],s[name+"_max"],color=engine_colors[unit],alpha=.1)
        ax.set(title=name+" — medyan ve uçuş içi min–maks.",xlabel="cycle — uçuş döngüsü",ylabel="İç sağlık parametresi — özgün değer")
        ax.legend(ncol=3)
    fig.suptitle("İşaret ve ölçek korunur; ilk ≠0 değer arıza başlangıcı veya kayıp yüzdesi değildir")
    return fig
finish(health_matrix_figure())
finish(active_health_figure())


## 6. Bütün sensörler: fiziksel ve sanal sensörler ayrı genel bakışlarda
Her panel yalnızca bir sensörü gösterir; farklı birimler tek Y ekseninde karıştırılmaz. Çizgiler uçuş medyanlarını, saydam bantlar Q1–Q3'ü gösterir; bantlar güven aralığı değildir. Her motorda aynı sensör için özgün veri ölçeği kullanılır. HDF5 değişken adı metadata'sında fiziksel birimler doğrulanmadığından tahmini birim atanmamıştır.

Bu grafiklerde bütün uçuşların tüm ölçümlerinden hesaplanan özetler kullanılır. Özetleme veri silme değildir. Çalışma koşullarının etkisi giderilmemiştir; görsel farklılık otomatik bozulma veya sensör önemi olarak yorumlanamaz.

In [ ]:
def draw_sensor(ax,name,x="cycle",band=True):
    for unit in ENGINES:
        s=engine_series(unit)
        ax.plot(s[x],s[name+"_median"],color=engine_colors[unit],linestyle=engine_styles[unit],linewidth=1,label=f"Motor {unit}")
        if band:
            ax.fill_between(s[x],s[name+"_q1"],s[name+"_q3"],color=engine_colors[unit],alpha=.06)
    ax.set_title(name)
    ax.set_ylabel(name+" — özgün değer",fontsize=9)
    ax.tick_params(labelsize=8)

def sensor_group_figure(group,label):
    fig,axes=plt.subplots(7,2,figsize=(18,23),sharex=True,sharey=False,layout="constrained")
    for ax,name in zip(axes.flat,group): draw_sensor(ax,name)
    handles,labels=axes[0,0].get_legend_handles_labels()
    axes[0,0].legend(handles,labels,loc="upper right",ncol=3,fontsize=8,framealpha=0.95)
    fig.suptitle(f"DS02 dev — {label}: medyan ve Q1–Q3 (tüm ölçümler)",fontsize=15)
    fig.supxlabel("cycle — uçuş döngüsü")
    return fig

def sensor_detail_figure(name,normalized=False):
    if name not in sensors: raise ValueError(name)
    fig,axes=plt.subplots(2 if normalized else 1,1,figsize=(14,9 if normalized else 5),squeeze=False,layout="constrained")
    draw_sensor(axes[0,0],name)
    axes[0,0].set_xlabel("cycle — uçuş döngüsü")
    axes[0,0].legend(ncol=3)
    if normalized:
        draw_sensor(axes[1,0],name,"life_progress")
        axes[1,0].set_xlabel("Normalize yaşam ilerlemesi — görsel hizalama; model girdisi değil")
    fig.suptitle(name+" — altı motor; çizgi: medyan, bant: Q1–Q3; koşul etkileri giderilmedi")
    return fig
finish(sensor_group_figure(physical_sensors,"14 fiziksel sensör"))
finish(sensor_group_figure(virtual_sensors,"14 sanal sensör"))
finish(sensor_detail_figure(sensors[0],True))


## 7. Çalışma koşulları ve Fc dağılımları
Koşul medyanları/Q1–Q3 bütün ölçümlerden hesaplanır. Fc dağılımı ölçüm ağırlıklı olarak gösterilir; detay tablosunda bir sınıfı içeren uçuş sayısı da vardır. Fc'nin sabit olması diğer dört koşulun da eşit olduğu anlamına gelmez. Aynı Fc sınıfındaki motorlarda bile uçuş profili, gaz kolu ve sıcaklık gibi koşullar farklılaşabilir. Bu nedenle ham sensör medyanları motorlar arasında birebir bir sağlık karşılaştırması sağlamaz.

In [ ]:
def conditions_figure():
    fig,axes=plt.subplots(2,2,figsize=(16,10),layout="constrained")
    for ax,name in zip(axes.flat,conditions):
        draw_sensor(ax,name)
        ax.set_xlabel("cycle — uçuş döngüsü")
    handles,labels=axes[0,0].get_legend_handles_labels()
    axes[0,0].legend(handles,labels,loc="upper right",ncol=3,fontsize=8,framealpha=0.95)
    fig.suptitle("DS02 dev — çalışma koşulları: medyan/Q1–Q3; bütün ölçümler")
    return fig

def fc_figure():
    fc=class_distribution.loc[class_distribution.variable=="Fc"].pivot(index="unit",columns="value",values="n_measurements").reindex(ENGINES).fillna(0)
    proportions=fc.div(fc.sum(axis=1),axis=0)*100
    fig,ax=plt.subplots(figsize=(12,5),layout="constrained")
    bottom=np.zeros(len(ENGINES))
    for value in proportions:
        ax.bar([str(u) for u in ENGINES],proportions[value],bottom=bottom,label=f"Fc {value:g}")
        bottom+=proportions[value].to_numpy()
    ax.set(xlabel="Motor ID",ylabel="Ölçüm payı (%)",title="Fc sınıfları — ölçüm ağırlıklı dağılım (uçuş sayısı değil)")
    ax.legend();return fig
finish(conditions_figure())
finish(fc_figure())
# Koşulların uçuş medyanı aralıkları: birleştirilmiş ham ölçüm kuantili değildir.
condition_ranges=pd.DataFrame([{"unit":unit,"condition":name,
    "flight_median_min":engine_series(unit)[name+"_median"].min(),
    "flight_median_max":engine_series(unit)[name+"_median"].max(),
    "median_of_flight_medians":engine_series(unit)[name+"_median"].median()}
    for unit in ENGINES for name in conditions])
scroll_table(condition_ranges)


## 8. Sensör seçimi — yalnızca mevcut uçuş özetlerinden
Başlangıç ayrıntılı grafik yukarıda kayıtlıdır. Bir sensör seçmek veya normalize eksen kutusunu değiştirmek altı motorun ayrıntılı karşılaştırmasını yeniler. Bu kontrol sensör elemesi yapmaz; bütün 28 sensör önceki genel bakışlarda korunur.

In [ ]:
sensor_select=widgets.Dropdown(options=[(f"{name} — {'fiziksel' if name in physical_sensors else 'sanal'}",name) for name in sensors],description="Sensör:")
normalize_axis=widgets.Checkbox(value=True,description="Normalize yaşam eksenini ayrıca göster",indent=False)
sensor_output=widgets.Output()
def update_sensor(change=None):
    with sensor_output:
        clear_output(wait=True)
        finish(sensor_detail_figure(sensor_select.value,normalize_axis.value))
sensor_select.observe(update_sensor,names="value")
normalize_axis.observe(update_sensor,names="value")
display(widgets.VBox([sensor_select,normalize_axis,sensor_output]))


## 9. Çıktılar ve doğrulama
CSV'ler yeni bir `outputs/DS02_cross_engine_.../` klasörüne kaydedilir. Uçuş özeti tüm 446 uçuşu ve sensör/koşul Q1-medyan-Q3 değerlerini içerir. Sağlık, Fc/hs, kalite ve kontrol tabloları ayrı dosyalardır. Önceki çıktı klasörleri değiştirilmez.

In [ ]:
from datetime import datetime
from uuid import uuid4
output_folder=project_root/"outputs"/f"DS02_cross_engine_{datetime.now():%Y%m%d_%H%M%S}_{uuid4().hex[:8]}"
output_folder.mkdir(parents=True,exist_ok=False)
outputs={"engine_summary":engine_summary,"flight_summary":flight_summary,"health_summary":health_summary,
         "class_distribution":class_distribution,"variable_quality":quality_summary,"checks":issues,"condition_ranges":condition_ranges}
for name,table in outputs.items():
    table.to_csv(output_folder/(name+".csv"),index=False,encoding="utf-8-sig")
print("CSV klasörü:",output_folder)
print(engine_summary[["unit","n_flights","n_measurements","first_RUL","last_RUL","nonzero_health_parameters"]].to_string(index=False))
print("Toplam dev ölçümü:",int(engine_summary.n_measurements.sum()))
print("RUL sabitlik:",int(engine_summary.RUL_constant_flights.sum()),"/",len(flight_summary))
print("Sonlu olmayan değer:",int(quality_summary.nonfinite_count.sum()))


## 10. Akademik değerlendirme — yalnızca hesaplanmış bulgular
Aşağıdaki değerlendirme, bu çalıştırmada elde edilen özetlerden üretilir. Motorların bozulma senaryoları önceden varsayılmaz. İç sağlık parametrelerinin farklılığı betimlenir; sensörlerin hangi fiziksel arızaya tepki verdiği veya arıza şiddeti hakkında doğrulanmamış karar verilmez.

In [ ]:
lines=["### Gerçek veriden elde edilen karşılaştırma",
    f"Altı dev motorunda {len(flight_summary)} uçuş ve {int(engine_summary.n_measurements.sum()):,} ölçüm kapsandı; dev satır toplamı ile tam eşleşti."]
for row in engine_summary.itertuples():
    h=health_summary.loc[(health_summary.unit==row.unit)&(health_summary.nonzero_samples>0)]
    detail="; ".join(f"{r.parameter}: {r.status}, ilk ≠0 cycle {r.first_nonzero_cycle}, ilk/son uçuş medyanı {r.first_flight_median:.6g} / {r.last_flight_median:.6g}" for r in h.itertuples()) or "sıfırdan farklı parametre yok"
    lines.append(f"- **Motor {row.unit}:** {row.n_flights} uçuş, {row.n_measurements:,} ölçüm; ilk/son RUL {row.first_RUL:g}/{row.last_RUL:g}. {detail}.")
lines += [f"Motor ömürleri {engine_summary.n_flights.min()}–{engine_summary.n_flights.max()} uçuş arasında değişiyor. Satır sayısı uçuş sayısından ayrı bir veri hacmi göstergesidir.",
    f"RUL uçuş içinde sabit/eksiksiz olarak doğrulanan uçuş sayısı: {int(engine_summary.RUL_constant_flights.sum())}/{len(flight_summary)}. Sonlu olmayan değer sayısı: {int(quality_summary.nonfinite_count.sum())}.",
    "### Ortak özellikler ve yorum sınırları",
    "Bütün motorlar aynı 28 sensör, 10 iç sağlık parametresi ve dört koşul değişkeniyle temsil ediliyor. Ortak isimler veya Fc sınıfı, koşul dağılımlarının eşit olduğunu göstermiyor. Koşulların motor bazında uçuş medyanı aralıkları yukarıdaki tabloda raporlandı.",
    "Sensör panellerindeki benzerlik/farklar ham medyan ve uçuş içi Q1–Q3 düzeyindedir. Ayrıntı kontrolü aynı sensörü bütün motorlarda aynı birim/özgün ölçek içinde karşılaştırır. Bu gösterim sensörlerin eşdeğer sağlık durumlarını ölçtüğünü, belirli arızaları ayırdığını veya bir sensörün daha iyi olduğunu doğrulamaz.",
    "İç sağlık eğrilerinin işaret ve ölçekleri korunmuştur. Değişken, sabit sıfır dışı ve sabit sıfır durumları ayrıdır; ilk ≠0 ölçüm arıza başlangıcı değildir. RUL eğrileri hedefin tanımını betimler; model başarısı göstermez.",
    "### RUL modelinden önce cevaplanması gereken sorular",
    "- Çalışma rejimi farkları sensör değişiminden nasıl ayrılacak; koşul karşılaştırılabilirliği nasıl doğrulanacak?",
    "- Hangi motorlar ve sağlık senaryoları arasında genelleme hedefleniyor; motor düzeyinde bağımsız değerlendirme nasıl tasarlanacak?",
    "- Sensör birimleri, örnekleme bilgisi ve sağlık parametrelerinin fiziksel anlamları hangi resmi metadata ile doğrulanacak?",
    "- Uçuş içi örnek bağımlılığı ve çok farklı uçuş uzunlukları nasıl ele alınacak?",
    "- Simülasyon iç sağlık değişkenlerinin gelecekte modele sızması nasıl önlenecek? Burada bunlar yalnızca EDA/doğrulama içindir.",
    "Bu notebook'ta bu sorular için model, veri bölme, ölçekleme, koşul filtresi veya özellik seçimi uygulanmadı."]
# Betimleyici örnekler: ilk/son uçuş medyanı farkı bütün eğrinin eğimi değildir.
examples=[]
for name in ["T24", "SmLPC"]:
    increased, decreased, equal = [], [], []
    for unit in ENGINES:
        values=engine_series(unit)[name+"_median"]
        difference=values.iloc[-1]-values.iloc[0]
        (increased if difference>0 else decreased if difference<0 else equal).append(unit)
    examples.append(f"- {name}: son uçuş medyanı ilkinden yüksek olan motorlar {increased}; düşük olanlar {decreased}; eşit olanlar {equal}.")
lines += ["### Sensör davranışına iki somut, betimleyici örnek"] + examples + [
    "Bu ilk/son karşılaştırmaları bütün eğrinin monoton olduğunu göstermez; uçuş medyanı ve Q1–Q3 panelleri birlikte okunmalıdır. Farklı koşullar ve uçuş profilleri etkili olabilir; yön farkı bir arıza sınıflandırması veya sensör önem sırası değildir.",
    "Bütün motorlarda ölçülen Fc sınıfı: " + ", ".join(str(v) for v in sorted(class_distribution.loc[class_distribution.variable=="Fc","value"].unique())) + ". Aynı Fc'ye rağmen koşul grafikleri ayrıca incelenmelidir."]
display(Markdown((chr(10)*2).join(lines)))
